In [1]:
#build the mlp. Keep it simple for now. 32

import numpy as np
import random

class SimpleMLP:
    def __init__(self, input_size, hidden_size, output_size):
        self.weights_input_hidden = np.random.randn(input_size, hidden_size) * 0.01  #change the inital weight activation if problems in training
        self.bias_hidden = np.zeros((1, hidden_size))
        self.weights_hidden_output = np.random.randn(hidden_size, output_size) * 0.01
        self.bias_output = np.zeros((1, output_size))

    def forward_pass(self, text):
        self.text = text  # store for backprop
        z1 = np.dot(text, self.weights_input_hidden) + self.bias_hidden
        self.a1 = self.ReLu(z1)
        z2 = np.dot(self.a1, self.weights_hidden_output) + self.bias_output
        output = self.sigmoid(z2)
        return output, self.a1 
    
    def ReLu(self, x):
        return np.maximum(0, x)
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def compute_loss(self, output, target):
        eps=1e-15 #to avoid error log(0)
        loss = -np.mean(target * np.log(output + eps) + (1 - target) * np.log(1 - output + eps))
        return loss

    def backward_pass(self, output, target):
        dz2 = output - target
        dw2 = np.dot(self.a1.T, dz2) / target.shape[0]
        db2 = np.sum(dz2, axis=0, keepdims=True) / target.shape[0]
        dz1 = np.dot(dz2, self.weights_hidden_output.T) * (self.a1 > 0)
        dw1 = np.dot(self.text.T, dz1) / target.shape[0]
        db1 = np.sum(dz1, axis=0, keepdims=True) / target.shape[0]
        return dw1, db1, dw2, db2

        

    def update_weights(self, dw1, db1, dw2, db2, learning_rate):
        self.weights_input_hidden -= learning_rate * dw1
        self.bias_hidden -= learning_rate * db1
        self.weights_hidden_output -= learning_rate * dw2
        self.bias_output -= learning_rate * db2

Test the MLP model works by gradient checking using dummy data.

In [13]:
#test everything works 
np.random.seed(30)

dummy_input = np.random.rand(3, 4)
dummy_target = np.array([[1], [0], [1]])

trial = SimpleMLP(input_size=4, hidden_size=6, output_size=1)
output = trial.forward_pass(dummy_input)[0]
print(output.shape)  # Should be (3, 1)
loss = trial.compute_loss(output, dummy_target)
print(loss)  # Should be a positive number
dw1, db1, dw2, db2 = trial.backward_pass(output, dummy_target)

# test hidden input weights
eps = 1e-5
start_value = trial.weights_input_hidden[0, 0] 

trial.weights_input_hidden[0, 0] = start_value + eps #slight positive change to weight
output_plus = trial.forward_pass(dummy_input)[0]
loss_plus = trial.compute_loss(output_plus, dummy_target)
trial.weights_input_hidden[0, 0] = start_value - eps #negative change to weight
output_minus = trial.forward_pass(dummy_input)[0]
loss_minus = trial.compute_loss(output_minus, dummy_target)

trial.weights_input_hidden[0, 0] = start_value #reset weight to original

numerical_grad = (loss_plus - loss_minus) / (2 * eps)
print("Numerical Gradient:", numerical_grad)
print("Analytic Gradient:", dw1[0, 0])  

# test hidden output weights
start_value_h = trial.weights_hidden_output[0, 0]

trial.weights_hidden_output[0, 0] = start_value_h + eps #slight positive change to weight
output_plus = trial.forward_pass(dummy_input)[0]
loss_plus = trial.compute_loss(output_plus, dummy_target)
trial.weights_hidden_output[0, 0] = start_value_h - eps #negative change to weight
output_minus = trial.forward_pass(dummy_input)[0]
loss_minus = trial.compute_loss(output_minus, dummy_target)

trial.weights_hidden_output[0, 0] = start_value_h #reset weight to original

numerical_grad = (loss_plus - loss_minus) / (2 * eps)
print("Numerical Gradient:", numerical_grad)
print("Analytic Gradient:", dw2[0, 0])  

# test hidden bias
start_value_bh = trial.bias_hidden[0, 3]

trial.bias_hidden[0, 3] = start_value_bh + eps #slight positive change to weight
output_plus = trial.forward_pass(dummy_input)[0]
loss_plus = trial.compute_loss(output_plus, dummy_target)
trial.bias_hidden[0, 3] = start_value_bh - eps #negative change to weight
output_minus = trial.forward_pass(dummy_input)[0]
loss_minus = trial.compute_loss(output_minus, dummy_target)

trial.bias_hidden[0, 3] = start_value_bh #reset weight to original

numerical_grad = (loss_plus - loss_minus) / (2 * eps)
print("Numerical Gradient:", numerical_grad)
print("Analytic Gradient:", db1[0, 3])  

# test output bias 
start_value_bo = trial.bias_output[0, 0]

trial.bias_output[0, 0] = start_value_bo + eps #slight positive change to weight
output_plus = trial.forward_pass(dummy_input)[0]
loss_plus = trial.compute_loss(output_plus, dummy_target)
trial.bias_output[0, 0] = start_value_bo - eps #negative change to weight
output_minus = trial.forward_pass(dummy_input)[0]
loss_minus = trial.compute_loss(output_minus, dummy_target)

trial.bias_output[0, 0] = start_value_bo #reset weight to original

numerical_grad = (loss_plus - loss_minus) / (2 * eps)
print("Numerical Gradient:", numerical_grad)
print("Analytic Gradient:", db2[0, 0])  

(3, 1)
0.6931538937595167
Numerical Gradient: 0.0003074773213818105
Analytic Gradient: 0.0003074773283330996
Numerical Gradient: -0.0014058584185683285
Analytic Gradient: -0.0014058584172150087
Numerical Gradient: 1.4866718966999313e-06
Analytic Gradient: 1.4866718518457649e-06
Numerical Gradient: -0.16675987633552047
Analytic Gradient: -0.16675987634032197
